# Model Comparison, Error Analysis & Model Selection

So sánh tất cả model (baseline + tuned), phân tích lỗi, chọn model tốt nhất.

**Ghi chú:** Notebook này cần kết quả từ:
- Notebooks 02–07 (baseline models) — chạy thủ công, kết quả nhập tay
- Notebook 08 (tuned models) — load từ `results/tuned_results.pkl`

In [ ]:
import sys
import pickle
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix

# Thêm thư mục gốc dự án vào sys.path
sys.path.insert(0, str(Path.cwd()))

from src.model_training import load_splits

# Matplotlib style
plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
})

In [ ]:
# --- STEP 1: Thu thập kết quả baseline ---
print("=" * 70)
print("STEP 1: Baseline Model Results")
print("=" * 70)

# Kết quả baseline từ notebooks 02-07 (nhập từ output đã chạy)
baseline_data = {
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "Extra Trees",
    ],
    "Accuracy": [0.9762, 0.9983, 0.9982, 0.9987, 0.9989, 0.9978],
    "Precision (weighted)": [0.9785, 0.9983, 0.9981, 0.9988, 0.9988, 0.9978],
    "Precision (macro)": [0.6386, 0.8814, 0.8547, 0.8386, 0.8494, 0.8369],
    "Recall (weighted)": [0.9762, 0.9983, 0.9982, 0.9987, 0.9989, 0.9978],
    "Recall (macro)": [0.5653, 0.7511, 0.8108, 0.8863, 0.8218, 0.7993],
    "F1 (weighted)": [0.9762, 0.9983, 0.9982, 0.9987, 0.9989, 0.9978],
    "F1 (macro)": [0.567, 0.7842, 0.8268, 0.8585, 0.8297, 0.813],
    "Cohen Kappa": [0.9214, 0.9944, 0.9939, 0.9957, 0.9962, 0.9927],
    "MCC": [0.9218, 0.9944, 0.9939, 0.9957, 0.9962, 0.9927],
    "ROC-AUC (weighted)": [0.997, 0.9988, 0.9998, 1.0, 0.999, 0.999],
    "ROC-AUC (macro)": [0.9762, 0.9552, 0.9866, 1.0, 0.9981, 0.9963],
    "Training Time (s)": [113.92, 72.65, 277.64, 112.84, 69.56, 81.00],
}

df_baseline = pd.DataFrame(baseline_data)
print(df_baseline.to_string(index=False))

In [ ]:
# --- STEP 2: Load kết quả tuned ---
print("\n" + "=" * 70)
print("STEP 2: Tuned Model Results")
print("=" * 70)

results_path = Path.cwd() / "results" / "tuned_results.pkl"

if results_path.exists():
    with open(results_path, "rb") as f:
        tuned_data = pickle.load(f)

    df_tuned = pd.DataFrame(tuned_data["all_results"])
    print(df_tuned.to_string(index=False))

    print("\nBest hyperparameters:")
    for model, params in tuned_data["best_params"].items():
        print(f"\n  {model}:")
        for k, v in params.items():
            print(f"    {k}: {v}")
else:
    print("⚠ tuned_results.pkl not found — run notebook 08 first!")
    df_tuned = pd.DataFrame()

In [ ]:
# --- STEP 3: So sánh Baseline vs Tuned ---
print("\n" + "=" * 70)
print("STEP 3: Baseline vs Tuned Comparison")
print("=" * 70)

# Chỉ so sánh 3 models đã tune
tuned_models = ["Random Forest", "XGBoost", "LightGBM"]
key_metrics = ["F1 (macro)", "F1 (weighted)", "Recall (macro)", "Precision (macro)", "ROC-AUC (macro)"]

comparison_rows = []
for model in tuned_models:
    baseline_row = df_baseline[df_baseline["Model"] == model].iloc[0]

    if not df_tuned.empty:
        tuned_row = df_tuned[df_tuned["Model"].str.contains(model)].iloc[0]
    else:
        tuned_row = None

    for metric in key_metrics:
        row = {"Model": model, "Metric": metric, "Baseline": baseline_row[metric]}
        if tuned_row is not None and metric in tuned_row:
            row["Tuned"] = tuned_row[metric]
            row["Δ"] = tuned_row[metric] - baseline_row[metric]
        comparison_rows.append(row)

df_compare = pd.DataFrame(comparison_rows)
print(df_compare.to_string(index=False))

In [ ]:
# --- STEP 4: Visualization — F1 Macro comparison ---
print("\n" + "=" * 70)
print("STEP 4: F1-Macro Comparison Chart")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: F1 macro cho tất cả baseline models
ax1 = axes[0]
models_sorted = df_baseline.sort_values("F1 (macro)", ascending=True)
colors = ["#e74c3c" if v < 0.7 else "#f39c12" if v < 0.8 else "#2ecc71"
          for v in models_sorted["F1 (macro)"]]
ax1.barh(models_sorted["Model"], models_sorted["F1 (macro)"], color=colors)
ax1.set_xlabel("F1 Score (macro)")
ax1.set_title("Baseline: F1-Macro by Model")
ax1.set_xlim(0.4, 1.0)
for i, (v, model) in enumerate(zip(models_sorted["F1 (macro)"], models_sorted["Model"])):
    ax1.text(v + 0.005, i, f"{v:.4f}", va="center", fontsize=9)

# Chart 2: Baseline vs Tuned (nếu có)
ax2 = axes[1]
if not df_tuned.empty:
    x = np.arange(len(tuned_models))
    width = 0.35
    baseline_vals = [df_baseline[df_baseline["Model"] == m]["F1 (macro)"].values[0] for m in tuned_models]
    tuned_vals = [df_tuned[df_tuned["Model"].str.contains(m)]["F1 (macro)"].values[0] for m in tuned_models]

    bars1 = ax2.bar(x - width/2, baseline_vals, width, label="Baseline", color="#3498db")
    bars2 = ax2.bar(x + width/2, tuned_vals, width, label="Tuned", color="#e67e22")

    ax2.set_xticks(x)
    ax2.set_xticklabels(tuned_models)
    ax2.set_ylabel("F1 Score (macro)")
    ax2.set_title("Baseline vs Tuned: F1-Macro")
    ax2.legend()
    ax2.set_ylim(0.7, 1.0)

    for bar, val in zip(bars1, baseline_vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f"{val:.4f}", ha="center", fontsize=8)
    for bar, val in zip(bars2, tuned_vals):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f"{val:.4f}", ha="center", fontsize=8)
else:
    ax2.text(0.5, 0.5, "Run notebook 08 first", ha="center", va="center",
             transform=ax2.transAxes, fontsize=14, color="gray")
    ax2.set_title("Baseline vs Tuned: F1-Macro")

plt.tight_layout()
plt.show()

In [ ]:
# --- STEP 5: Per-Class Analysis ---
print("\n" + "=" * 70)
print("STEP 5: Per-Class Performance Analysis")
print("=" * 70)

# Load test data để tính per-class metrics
X_train, X_test, y_train, y_test = load_splits()
classes = sorted(y_test.unique())

print(f"Classes: {classes}")
print(f"Test set size: {len(y_test):,}")
print(f"\nClass distribution in test set:")
for cls in classes:
    cnt = (y_test == cls).sum()
    print(f"  {cls:35s} {cnt:>8,} ({cnt/len(y_test)*100:6.2f}%)")

In [ ]:
# --- STEP 6: Per-Class F1 Heatmap ---
print("\n" + "=" * 70)
print("STEP 6: Per-Class F1 Score Heatmap")
print("=" * 70)

# Chạy lại predictions cho các model tốt nhất
# Load tuned models từ kết quả notebook 08
# (Nếu chưa có tuned, chỉ dùng baseline)

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import lightgbm as lgbm
import time

# Train lại các model (hoặc load nếu đã save)
# Để tiết kiệm thời gian, ta chỉ train model tốt nhất
# Ở đây sẽ train baseline của tất cả models

models_config = {
    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1000, solver="lbfgs", random_state=42, n_jobs=-1),
        "needs_scaling": True,
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(max_depth=20, random_state=42),
        "needs_scaling": False,
    },
    "Random Forest": {
        "model": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        "needs_scaling": False,
    },
    "XGBoost": {
        "model": None,  # cần label encoding riêng
        "needs_scaling": False,
    },
    "LightGBM": {
        "model": lgbm.LGBMClassifier(
            num_leaves=31, learning_rate=0.05, n_estimators=200,
            subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1,
            random_state=42, n_jobs=-1, verbose=-1,
        ),
        "needs_scaling": False,
    },
    "Extra Trees": {
        "model": ExtraTreesClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        "needs_scaling": False,
    },
}

# Scaler cho LR
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Label encoder cho XGBoost
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)

# Train tất cả và lấy per-class report
per_class_f1 = {}

for name, config in models_config.items():
    print(f"\nTraining {name}...")
    t0 = time.time()

    if name == "XGBoost":
        weights = compute_sample_weight(class_weight="balanced", y=y_train_enc)
        model = XGBClassifier(
            objective="multi:softprob", n_estimators=300, max_depth=7,
            learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
            random_state=42, n_jobs=-1, verbosity=0, eval_metric="mlogloss",
        )
        model.fit(X_train, y_train_enc, sample_weight=weights)
        y_pred_enc = model.predict(X_test)
        y_pred = le.inverse_transform(y_pred_enc)
    elif config["needs_scaling"]:
        model = config["model"]
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model = config["model"]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    elapsed = time.time() - t0
    print(f"  Done in {elapsed:.1f}s")

    # Per-class F1
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    per_class_f1[name] = {cls: report[cls]["f1-score"] for cls in classes if cls in report}

# Tạo heatmap DataFrame
df_perclass = pd.DataFrame(per_class_f1).T  # rows = models, cols = classes
df_perclass = df_perclass[classes]  # sắp xếp cột theo tên class

print("\nPer-class F1 scores:")
print(df_perclass.round(4).to_string())

In [ ]:
# Vẽ heatmap
fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    df_perclass,
    annot=True, fmt=".3f",
    cmap="RdYlGn", vmin=0, vmax=1,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Per-Class F1 Score by Model", fontsize=14, fontweight="bold")
ax.set_xlabel("Attack Class")
ax.set_ylabel("Model")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# --- STEP 7: Error Analysis ---
print("\n" + "=" * 70)
print("STEP 7: Error Analysis — Top Misclassification Pairs")
print("=" * 70)

# Phân tích lỗi cho model tốt nhất (XGBoost baseline dựa trên F1 macro)
# Dùng lại y_pred từ XGBoost ở trên
best_model_name = "XGBoost"

# Lấy lại predictions cho XGBoost
weights = compute_sample_weight(class_weight="balanced", y=y_train_enc)
xgb_best = XGBClassifier(
    objective="multi:softprob", n_estimators=300, max_depth=7,
    learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0, eval_metric="mlogloss",
)
xgb_best.fit(X_train, y_train_enc, sample_weight=weights)
y_pred_best_enc = xgb_best.predict(X_test)
y_pred_best = le.inverse_transform(y_pred_best_enc)

# Tìm top misclassification pairs
cm = confusion_matrix(y_test, y_pred_best, labels=classes)
misclass_pairs = []
for i, true_cls in enumerate(classes):
    for j, pred_cls in enumerate(classes):
        if i != j and cm[i, j] > 0:
            misclass_pairs.append({
                "True Class": true_cls,
                "Predicted As": pred_cls,
                "Count": cm[i, j],
                "% of True Class": round(cm[i, j] / cm[i].sum() * 100, 2),
            })

df_errors = pd.DataFrame(misclass_pairs).sort_values("Count", ascending=False)
print(f"\nTop 15 misclassification pairs ({best_model_name}):")
print(df_errors.head(15).to_string(index=False))

In [ ]:
# Vẽ top misclassifications
top_errors = df_errors.head(10).copy()
top_errors["Pair"] = top_errors["True Class"] + " → " + top_errors["Predicted As"]

fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#e74c3c" if pct > 5 else "#f39c12" if pct > 1 else "#3498db"
          for pct in top_errors["% of True Class"]]
ax.barh(top_errors["Pair"][::-1], top_errors["Count"][::-1], color=colors[::-1])
ax.set_xlabel("Number of Misclassifications")
ax.set_title(f"Top 10 Misclassification Pairs — {best_model_name}", fontsize=13, fontweight="bold")

for i, (cnt, pct) in enumerate(zip(top_errors["Count"][::-1], top_errors["% of True Class"][::-1])):
    ax.text(cnt + 5, i, f"{cnt:,} ({pct}%)", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## Model Selection — Kết luận

### Tiêu chí chọn model

| Tiêu chí | Metric | Ý nghĩa |
|----------|--------|---------|
| **Phát hiện tốt minority attacks** | F1-macro | Tất cả class được đánh giá công bằng |
| **Độ chính xác tổng thể** | F1-weighted, Accuracy | Quan trọng cho production |
| **Phân biệt class tốt** | ROC-AUC macro | Khả năng ranking probability |
| **Tốc độ** | Training time | Quan trọng nếu cần retrain thường xuyên |

In [ ]:
# --- STEP 8: Final Model Recommendation ---
print("=" * 70)
print("STEP 8: Final Model Recommendation")
print("=" * 70)

# Tổng hợp và ranking
ranking_metrics = ["F1 (macro)", "F1 (weighted)", "ROC-AUC (macro)", "MCC"]

# Ranking cho mỗi metric (1 = best)
rankings = pd.DataFrame({"Model": df_baseline["Model"]})
for metric in ranking_metrics:
    rankings[metric + " rank"] = df_baseline[metric].rank(ascending=False).astype(int)

rankings["Avg Rank"] = rankings[[c for c in rankings.columns if "rank" in c]].mean(axis=1)
rankings = rankings.sort_values("Avg Rank")

print("\nModel Rankings (1 = best):")
print(rankings.to_string(index=False))

# Kết luận
best = rankings.iloc[0]["Model"]
print(f"\n{'='*70}")
print(f"  ★ RECOMMENDED MODEL: {best}")
print(f"{'='*70}")

best_row = df_baseline[df_baseline["Model"] == best].iloc[0]
print(f"\n  F1 (macro):    {best_row['F1 (macro)']}")
print(f"  F1 (weighted): {best_row['F1 (weighted)']}")
print(f"  ROC-AUC:       {best_row['ROC-AUC (macro)']}")
print(f"  MCC:           {best_row['MCC']}")
print(f"  Training time: {best_row['Training Time (s)']:.1f}s")

print("\n  Lưu ý:")
print("  - Heartbleed (9 samples) và SQL Injection (17 samples) quá ít")
print("    để đánh giá có ý nghĩa thống kê.")
print("  - Kết quả tuned (notebook 08) có thể thay đổi ranking.")
print("  - Trong production nên dùng model tuned thay vì baseline.")